# Session 33 — Building a RAG Pipeline, End to End

### A step-by-step teaching notebook · fully practical

**How to use this notebook.** It is written for *you*, the instructor, to work through
**live** with the client. Every step follows the same rhythm:

| Part | What you do |
|---|---|
| 🎯 **What** | One sentence: the thing we are about to build |
| 💡 **Why** | The problem it solves — say this *before* typing |
| ⌨️ **Code** | Type it live. Only this step's code, nothing more |
| 🔍 **How it works** | Line by line |
| ⚙️ **Behind the scenes** | What actually happens at runtime |
| ❓ **Likely questions** | Prepared answers for what they will ask |
| ✅ **Checkpoint** | Pause. Ask. Do not move on until they answer |

**Where this sits.** Session 32b ended with a 10-sentence toy corpus and a working
`retrieve → prompt → answer` loop. Today we do the same thing on **real documents** —
three research PDFs, downloaded live — and add the four stages that separate a demo
from a system: **chunking, hybrid retrieval, reranking, and evaluation**.

**Path through the notebook**

```
Step 0   Setup + fetch the corpus            ─┐
Step 1   Ingestion — PDF to clean text        │  "how does a document get in?"
Step 2   Chunking — the highest-leverage knob ─┘

Step 3   Indexing — embed + FAISS + persist   ─┐
Step 4   Retrieval — dense search              │  "how do we find the right piece?"
Step 5   Hybrid retrieval — BM25 + RRF         │
Step 6   Reranking — the cross-encoder        ─┘

Step 7   Prompt construction — grounding & citations
Step 8   Evaluation — does any of this actually help?
Step 9   The whole pipeline in one function + failure drills
Step 10  Recap, the knobs table, exercises
```

**Before the session**

1. `pip install openai numpy faiss-cpu tiktoken pypdf rank_bm25 sentence-transformers`
2. A working `OPENAI_API_KEY` in your environment — run Step 0 once to confirm.
3. Run the whole notebook once **the day before**. Step 6 downloads a ~90 MB reranker
   model; you do not want that download happening live.
4. Total API cost for a full run: roughly 5–8 cents (`gpt-5.6-luna` at \$1 / \$6 per 1M
   tokens, `text-embedding-3-small` at \$0.02 per 1M).

> **Timing:** Steps 0–2 ≈ 30 min · Steps 3–6 ≈ 30 min · Steps 7–9 ≈ 25 min.
> If you are running short, **Step 5 (hybrid)** is the one to drop — Step 8 still works,
> it just compares two rows instead of four.

---

## Step 0 · Setup and the corpus

### 🎯 What we are going to implement

Install the libraries, confirm the API key works, and download three real research
papers as PDFs.

### 💡 Why we are implementing it

Session 32b used ten hand-written sentences. That was the right choice then — the client
could verify every search result with their own eyes. It is the wrong choice now, because
every hard problem in RAG comes from documents that are **long, messy, and not written for
you**. A PDF gives us all three for free: two-column layout, hyphens split across lines,
headers, footnotes, references.

The three papers are chosen deliberately — they are the papers *about* the thing we are
building, so the client can judge whether an answer is correct without being a domain expert.

In [1]:
# Run once per machine.
!pip install pypdf rank_bm25 sentence-transformers

  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/596.7 kB ? eta -:--:--
   ----------------------------------- ---- 524.3/596.7 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 596.7/596.7 kB 1.9 MB/s  0:00:00
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl (355 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ------- -------------------------------- 1.6/8.3 MB 6.8 MB/s eta 0:00:01
   -------------------- ----

In [ ]:
import os, re, json, time, textwrap, pathlib
import numpy as np
from openai import OpenAI

client = OpenAI()                              # reads OPENAI_API_KEY from the environment
EMBED_MODEL = "text-embedding-3-small"         # 1536 dimensions, same as Session 32b
CHAT_MODEL  = "gpt-5.6-luna"

# One cheap call proves the key is alive before we build anything on top of it.
print("key works:", len(client.embeddings.create(model=EMBED_MODEL, input="ping").data[0].embedding))

In [ ]:
import urllib.request

DATA = pathlib.Path("rag_corpus"); DATA.mkdir(exist_ok=True)

# Three papers, all open-access on arXiv. The label is what the model will cite.
SOURCES = {
    "rag_lewis_2020.pdf":           ("https://arxiv.org/pdf/2005.11401", "RAG (Lewis et al., 2020)"),
    "dpr_karpukhin_2020.pdf":       ("https://arxiv.org/pdf/2004.04906", "DPR (Karpukhin et al., 2020)"),
    "lost_in_the_middle_2023.pdf":  ("https://arxiv.org/pdf/2307.03172", "Lost in the Middle (Liu et al., 2023)"),
}

for name, (url, label) in SOURCES.items():
    path = DATA / name
    if not path.exists():                              # don't re-download on a re-run
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (teaching-notebook)"})
        path.write_bytes(urllib.request.urlopen(req, timeout=60).read())
    print(f"{name:32s} {path.stat().st_size/1024:8.0f} KB   {label}")

### 🔍 How the code works

- `OpenAI()` with no arguments reads `OPENAI_API_KEY` from the environment. If it is
  missing you get an `OpenAIError` on the **first API call** — which is exactly why the
  ping call is there, and why it is the first thing we run.
- `if not path.exists()` makes the cell **idempotent**. You will re-run cells all session;
  none of them should re-download 2 MB or re-spend money. Build this habit early.
- The `User-Agent` header is not decoration — arXiv rejects requests from the default
  Python user agent.
- Each source carries a **human-readable label** from the very beginning. That label
  travels with every chunk and ends up inside the citation the model prints in Step 7.
  Metadata you do not attach at ingestion time is metadata you cannot cite later.

### ⚙️ Behind the scenes

Nothing has been read yet — these are bytes on disk. About 2 MB of PDF, which will become
roughly 190,000 characters of text, roughly 47,000 tokens, and roughly 180 chunks.
Keep those four numbers in view; they are the shape of the whole session.

### ❓ Questions the client will ask

> *"Why not just paste all three papers into the prompt? `gpt-5.6-luna` has a million-token
> context window — 50k tokens is nothing."*
> This is the best question in the session and it deserves a real answer, because **the
> context window is no longer the constraint**. Three reasons remain. (1) Cost: ~30× more
> **per question**, forever, and it does not amortise. (2) Latency. (3) Accuracy — it is
> often *worse*, which is the surprising one and the entire subject of the third paper we
> just downloaded. We demonstrate it in Step 7. And none of the three scale: a real corpus
> is 50 million tokens, not 50 thousand.

> *"Does this work on Word documents / HTML / scanned pages?"*
> Yes — swap the loader in Step 1, everything downstream is unchanged. That separation is
> the point of having an ingestion stage at all. Scans need OCR first (the OCR session, PaddleOCR).

### ✅ Checkpoint

Ask: **"We have 2 MB of PDF and a question to answer. What has to happen in between?"**
Let them sketch it. Then show them their own answer is the notebook's table of contents.

---

## Step 1 · Ingestion — from PDF to clean text

### 🎯 What we are going to implement

Extract text page by page with `pypdf`, then write a cleaning function that repairs the
damage PDF extraction always does.

### 💡 Why we are implementing it

This is the least glamorous stage and the one that quietly decides how good the system can
possibly be. **Everything downstream sees only what ingestion produces.** If a sentence is
broken across a line, the embedding of that chunk is subtly wrong, the BM25 term match
fails, the reranker sees garbage — and none of it shows up as an error. It shows up as
"the retrieval is a bit rubbish" three weeks later.

Look at the raw output first, with the client, before you clean anything.

In [ ]:
from pypdf import PdfReader

reader = PdfReader(DATA / "rag_lewis_2020.pdf")
raw_page = reader.pages[0].extract_text()          # page 1: title, authors, abstract

print("pages in PDF :", len(reader.pages))
print("chars on p.1 :", len(raw_page))
print("-" * 78)
print(raw_page[380:1200])

### 🔍 What you are looking at

Read the output aloud with the client and name the damage:

1. **Hyphenation** — `down-\nstream`, `knowl-\nedge`, `Pre-\ntrained`, `mem-\nory`. Each was
   **one word** in the paper. It is now two fragments that match nothing and embed as noise.
2. **Hard line wraps** — every line of the PDF ends in `\n`, including mid-sentence. A
   paragraph is not a paragraph any more, so a splitter has no boundaries to respect.
3. **Ligature artefacts** — `ﬁne-tuned`, `speciﬁc`. That `ﬁ` is a *single* Unicode character,
   not `f` + `i`, so a keyword search for "fine-tuned" will miss it.
4. **Furniture** — page numbers, headers, footnote markers, all inline with the prose.
5. **Reading order** — two-column and figure-heavy pages can interleave. Run
   `print(reader.pages[1].extract_text()[:400])` if you want to show them a genuinely
   scrambled page; that one is Figure 1 and it comes out as word soup.

None of this raises an exception. That is the danger.

In [ ]:
def clean(text: str) -> str:
    """Repair the standard damage of PDF text extraction."""
    text = text.replace("­", "")                          # soft hyphens
    text = re.sub(r"([a-z])-\n([a-z])", r"\1\2", text)         # de-hyphenate across lines
    text = re.sub(r"(?<![.!?:;])\n(?![\n•\-\d])", " ", text)  # unwrap mid-sentence breaks
    text = re.sub(r"[ \t]+", " ", text)                        # collapse runs of spaces
    text = re.sub(r"\n{3,}", "\n\n", text)                     # collapse blank lines
    return text.strip()

print(clean(raw_page)[380:1000])

In [ ]:
def load_pdf(path: pathlib.Path, label: str):
    """PDF -> list of {source, page, text}. One record per page, cleaned."""
    pages = []
    for i, page in enumerate(PdfReader(path).pages, start=1):
        text = clean(page.extract_text() or "")
        if len(text) < 50:                 # skip title-only or figure-only pages
            continue
        pages.append({"source": label, "page": i, "text": text})
    return pages

PAGES = []
for name, (url, label) in SOURCES.items():
    got = load_pdf(DATA / name, label)
    PAGES.extend(got)
    print(f"{label:38s} {len(got):3d} pages kept, {sum(len(p['text']) for p in got):7,d} chars")

print("-" * 78)
print(f"{'TOTAL':38s} {len(PAGES):3d} pages, {sum(len(p['text']) for p in PAGES):7,d} chars")

### 🔍 How the code works

- **`([a-z])-\n([a-z])` → `\1\2`** joins `non-` + `parametric`. It requires lowercase on
  both sides so that `state-of-the-art` at a line break, or a hyphen after a capital, is
  left alone. Cheap heuristic, catches the large majority.
- **The unwrap regex** is the interesting one. `(?<![.!?:;])` is a *negative lookbehind*:
  only unwrap a newline that is **not** preceded by sentence-ending punctuation. So a wrapped
  sentence is rejoined, but a real paragraph break survives. `(?![\n•\-\d])` is a negative
  lookahead protecting blank lines and list items. Those surviving `\n\n` marks are what
  Step 2's splitter uses to find natural boundaries — cleaning and chunking are one design,
  not two.
- **`len(text) < 50`** drops pages that are a figure and nothing else. An almost-empty chunk
  is worse than no chunk: it still competes for a top-k slot.
- **The return shape** — `{source, page, text}` — is a deliberate contract. Every later stage
  consumes exactly this. Swap `load_pdf` for `load_html` and the rest of the notebook does
  not notice.

### ⚙️ Behind the scenes

`pypdf` reads the content stream and reconstructs text from glyph positions. There is no
"text" in a PDF — there are characters placed at coordinates. That is why extraction is
inherently lossy, and why heavier tools (`unstructured`, `PyMuPDF`, Docling, or a vision
model) exist for tables and complex layouts. Our three papers are prose, so `pypdf` is
plenty and installs in two seconds.

### ❓ Questions the client will ask

> *"Should we strip headers, footers, and the references section?"*
> Usually yes for headers and footers — they repeat on every page and dilute every chunk.
> References are a judgement call: useless for "what did they find?", essential for "who did
> they cite?". Decide from the questions you expect, not from taste.

> *"How do I know my cleaning is good enough?"*
> You do not, by inspection. You find out in Step 8. Ingestion changes are exactly the kind
> of change that needs a number attached, and that is what the evaluation harness is for.

### ✅ Checkpoint

Ask: **"We kept the page number on every record. Why bother?"**
Answer: it is the difference between *"the paper says X"* and *"the paper says X, page 6"* —
a citation the client can click. Trust in a RAG system is built almost entirely out of
verifiable citations.

---

## Step 2 · Chunking — the highest-leverage knob in the system

### 🎯 What we are going to implement

Two splitters: a naive fixed-character one, then a token-aware recursive one with overlap.
We compare them on the same page and keep the second.

### 💡 Why we are implementing it

**The chunk is the unit of retrieval.** Whatever a chunk is, that is the smallest thing your
system can find and the largest thing it can find *precisely*. Get this wrong and no
reranker, no better embedding model, and no smarter prompt will save you.

Two failure modes, pulling in opposite directions:

| Chunks too small | Chunks too large |
|---|---|
| The answer is split across two chunks; you retrieve half of it | One chunk covers five topics; its embedding is an average of all five |
| High precision, low recall | High recall, low precision |
| The model answers with a fragment | The model gets 800 tokens of noise around 20 tokens of signal |

The 2025–2026 consensus lands at **256–512 tokens with 10–20% overlap**, and recursive
splitting is the default in every production framework. We will use 320 and 48 — and then
in Step 8 the client gets to *check* whether that was a good choice.

In [ ]:
import tiktoken
ENC = tiktoken.get_encoding("cl100k_base")           # the tokenizer OpenAI models use
def ntok(s: str) -> int: return len(ENC.encode(s))

sample = PAGES[4]["text"]        # page 5 of the RAG paper — dense, ordinary prose
print("page tokens:", ntok(sample), " chars:", len(sample), " ratio:", round(len(sample)/ntok(sample), 2))

In [ ]:
# ── The naive way: cut every N characters. Nobody ships this; everybody starts here.
def split_naive(text, size=1100):
    return [text[i:i+size] for i in range(0, len(text), size)]

for i, c in enumerate(split_naive(sample)[:2]):
    print(f"--- naive chunk {i} ({ntok(c)} tokens) ---")
    print("START:", repr(c[:70]))
    print("END  :", repr(c[-70:]))

### 🔍 Look at where it cut

Chunk 0 ends `...generating Jeopardy ques` and chunk 1 begins `tions conditioned on...` —
the splitter cut **inside a word**, because character 1100 is where it happened to land.
Two problems, and the second is the serious one:

1. The fragment at the end of chunk 0 is noise in its embedding.
2. The fragment at the *start of chunk 1* has lost its subject. Neither chunk can answer a
   question about Jeopardy question generation on its own — and retrieval returns one chunk,
   not two.

Now the version we keep.

In [ ]:
SEPARATORS = ["\n\n", "\n", ". ", " "]      # most natural boundary first

def split_recursive(text, size=320, overlap=48):
    """Split to <= `size` tokens, preferring the most natural boundary available.

    Try paragraphs. Any piece still too big, retry on newlines, then sentences,
    then words, then a hard token cut. Finally, glue `overlap` tokens of the
    previous chunk onto the front of each chunk.
    """
    def rec(t, depth):
        if ntok(t) <= size:
            return [t]
        if depth >= len(SEPARATORS):                       # last resort: hard token window
            ids = ENC.encode(t)
            return [ENC.decode(ids[i:i+size]) for i in range(0, len(ids), size)]
        sep, out, buf = SEPARATORS[depth], [], ""
        for part in t.split(sep):
            cand = f"{buf}{sep}{part}" if buf else part
            if ntok(cand) <= size:
                buf = cand                                  # keep packing
            else:
                if buf: out.append(buf)                     # flush what we had
                if ntok(part) <= size:
                    buf = part
                else:
                    buf = ""
                    out.extend(rec(part, depth + 1))        # this part alone is too big
        if buf: out.append(buf)
        return out

    pieces = rec(text, 0)
    chunks = []
    for i, p in enumerate(pieces):
        if i and overlap:                                   # prepend the tail of the previous
            tail = ENC.decode(ENC.encode(pieces[i-1])[-overlap:])
            p = f"{tail} {p}"
        p = p.strip()
        if ntok(p) > 20:                                    # drop scraps
            chunks.append(p)
    return chunks

for i, c in enumerate(split_recursive(sample)[:2]):
    print(f"--- recursive chunk {i} ({ntok(c)} tokens) ---")
    print("START:", repr(c[:70]))
    print("END  :", repr(c[-70:]))

In [ ]:
# Apply it to every page and attach the metadata each chunk needs to be cited.
CHUNKS = []
for p in PAGES:
    for piece in split_recursive(p["text"]):
        CHUNKS.append({
            "id":     f"c{len(CHUNKS):04d}",
            "text":   piece,
            "source": p["source"],
            "page":   p["page"],
            "tokens": ntok(piece),
        })

lens = np.array([c["tokens"] for c in CHUNKS])
print(f"chunks       : {len(CHUNKS)}")
print(f"tokens       : min {lens.min()}  median {int(np.median(lens))}  max {lens.max()}")
print(f"total tokens : {lens.sum():,}   (embedding cost ~${lens.sum()/1e6*0.02:.4f})")
print()
print(json.dumps(CHUNKS[5], indent=2)[:600])

### 🔍 How the code works

- **`SEPARATORS` is a priority list, not a set.** Paragraph breaks first because they are the
  author's own idea of where a topic ends. We only fall to cruder boundaries when the text
  gives us nothing better. That is what "recursive" means — LangChain's
  `RecursiveCharacterTextSplitter` is this function with more edge cases.
- **`buf` is a greedy packer.** We keep appending parts while the result still fits, then
  flush. This is why chunks come out near the target size instead of tiny — a splitter that
  cuts at *every* separator wastes the budget.
- **`out.extend(rec(part, depth + 1))`** is the recursion: one paragraph that is on its own
  bigger than 320 tokens gets re-split on sentences instead.
- **`ntok(p) > 20`** drops scraps — a stray heading, a page number that survived cleaning.
- **The overlap** takes the last 48 tokens of the *previous* chunk and prepends them. A
  sentence that straddles a boundary now appears **whole** in the second chunk. Cost: about
  15% more vectors and 15% more storage. That is the trade.

### ⚙️ Behind the scenes

`ENC.encode(t)` is BPE tokenization — the same one the embedding model bills you for, which
is why counting tokens here and not characters keeps you honest. Note the chars-per-token
ratio you printed: ~4 for ordinary English, and it drops sharply for code, tables, or
non-English text. Never budget in characters.

### ❓ Questions the client will ask

> *"Is 320 the right number?"*
> It is a defensible starting point, not a right answer. It depends on the density of your
> documents and the shape of your questions — short factoid questions favour smaller chunks,
> "explain the method" questions favour larger. **Step 8 is how you decide**, and re-running
> this notebook with `size=192` or `size=640` is the exercise at the end.

> *"Isn't overlap just duplicated data?"*
> Yes, and it is not always worth it. A January 2026 analysis on Natural Questions found no
> measurable benefit from overlap at all on that dataset. Start at 10%, then measure. This is
> the recurring lesson of the session: RAG has no universal defaults, only measurable ones.

> *"What about semantic chunking — splitting where the meaning shifts?"*
> Real technique: embed sentence by sentence, cut where consecutive similarity drops. Buys a
> few points of recall for roughly one embedding call per sentence at index time. Worth it for
> a high-value corpus; usually not the first thing to reach for.

### ✅ Checkpoint

Ask: **"A number lives in a table on page 6. What does our chunker do to it, and is that a
problem?"** Answer: the table's rows get flattened into prose and probably split mid-table.
It is a real problem, and it is why table-aware loaders exist. Naming a limitation out loud
is more valuable than pretending the pipeline is general.

---

## Step 3 · Indexing — embed, store, persist

### 🎯 What we are going to implement

Embed all ~180 chunks in batches, load them into a FAISS index, and save **both** the index
and the chunk records to disk.

### 💡 Why we are implementing it

Session 32b covered the mechanics: embed, normalise, `IndexFlatIP`. Two things are new here,
and both are the kind of thing that only shows up at real scale.

1. **Batching.** 180 chunks in one request would exceed the input limit and be fragile
   besides. Batched requests with order-preservation are how this is actually done.
2. **The index does not store your text.** FAISS gives you back integer positions. If you
   lose the alignment between position *i* and chunk *i*, every citation in your system
   silently points at the wrong paragraph — and nothing crashes. **Save them together, always.**

In [ ]:
def embed(texts, model=EMBED_MODEL, batch=64):
    """Embed a list of strings -> (n, d) float32 array, input order preserved."""
    if isinstance(texts, str):
        texts = [texts]
    out = []
    for i in range(0, len(texts), batch):
        resp = client.embeddings.create(model=model, input=texts[i:i+batch])
        # Sort by .index — never assume the API returns results in input order.
        out.extend(r.embedding for r in sorted(resp.data, key=lambda r: r.index))
    return np.asarray(out, dtype="float32")

t0 = time.time()
E = embed([c["text"] for c in CHUNKS])
print(f"embedded {E.shape[0]} chunks -> {E.shape[1]} dims in {time.time()-t0:.1f}s")
print("norm of row 0:", round(float(np.linalg.norm(E[0])), 4), "(OpenAI returns unit vectors)")

In [ ]:
import faiss

index = faiss.IndexFlatIP(E.shape[1])     # inner product == cosine on unit vectors
Xb = E.copy()                             # copy: normalize_L2 works in place
faiss.normalize_L2(Xb)
index.add(Xb)

print("ntotal:", index.ntotal, "| dim:", index.d, "| chunks:", len(CHUNKS))
assert index.ntotal == len(CHUNKS), "index and chunk list are out of sync"

In [ ]:
# Persist the pair. Losing either half makes the other half useless.
faiss.write_index(index, "rag_papers.index")
pathlib.Path("rag_papers_chunks.json").write_text(json.dumps(CHUNKS), encoding="utf-8")

print("index  :", f"{os.path.getsize('rag_papers.index')/1e6:.2f} MB")
print("chunks :", f"{os.path.getsize('rag_papers_chunks.json')/1e6:.2f} MB")
print("rule of thumb: N x dims x 4 bytes =", f"{len(CHUNKS)*E.shape[1]*4/1e6:.2f} MB")

### 🔍 How the code works

- **`for i in range(0, len(texts), batch)`** walks the list in slices of 64. One request per
  slice instead of one per chunk: same tokens billed, a fraction of the round trips.
- **`sorted(resp.data, key=lambda r: r.index)`** is defensive and non-negotiable. The API
  returns an `index` field precisely so you can restore input order. Get this wrong and every
  chunk is paired with some other chunk's vector — a bug with no symptom except bad answers.
- **`assert index.ntotal == len(CHUNKS)`** is the cheapest bug-catcher in the notebook. Put an
  assert wherever two structures have to stay aligned.
- **`IndexFlatIP`** is exhaustive and exact. At 180 vectors — or 150,000 — that is the right
  choice. Session 32b's IVF index only starts to pay at millions.

### ⚙️ Behind the scenes

180 chunks × ~320 tokens ≈ 58,000 tokens ≈ **\$0.0012**. The index itself is
`180 × 1536 × 4 bytes ≈ 1.1 MB`. Say the general form out loud, because it is how you size a
real deployment: **N × dims × 4 bytes**. A million chunks at 1536 dims is 6 GB of RAM — which
is the moment IVF, HNSW, or a hosted vector database stops being optional.

### ❓ Questions the client will ask

> *"What happens when a document changes?"*
> You re-embed and re-index the chunks from that document. This is why the chunk `id` carries
> its source: a real system stores `doc_id` and `content_hash` per chunk, then re-processes
> only what moved. Rebuilding the whole index nightly works fine until it doesn't.

> *"Can I mix embedding models?"*
> No. Vectors from two different models are not comparable, even at the same dimension.
> Changing the embedding model means re-indexing everything.

### ✅ Checkpoint

Ask: **"I `write_index` but forget to save the chunk JSON. What breaks, and when do I find out?"**
Answer: search still returns ids and scores, so nothing crashes — you find out when a user
notices the citation is nonsense. Silent failures are the theme of this session.

---

## Step 4 · Retrieval — dense search over real documents

### 🎯 What we are going to implement

A `dense_search(query, k)` function, and a small display helper we will reuse for the rest of
the notebook.

### 💡 Why we are implementing it

Mechanically this is Session 32b again. What is new is the *evidence*: on ten hand-written
sentences dense retrieval looks flawless. On ~180 chunks from three papers you get to see it
succeed on paraphrase — and, in a moment, fail on exact terms. Both are the point.

In [ ]:
def dense_search(query, k=5):
    """Return [(score, chunk), ...] ranked by cosine similarity."""
    q = embed(query)                    # (1, d)
    faiss.normalize_L2(q)
    scores, ids = index.search(q, k)    # both (1, k)
    return [(float(s), CHUNKS[i]) for s, i in zip(scores[0], ids[0]) if i != -1]

def show(results, query=None, width=96):
    if query: print(f"Q: {query}\n" + "-" * width)
    for rank, (score, c) in enumerate(results, start=1):
        head = f"{rank}. {score:6.3f}  [{c['id']}] {c['source']} p{c['page']}"
        print(head)
        print(textwrap.indent(textwrap.fill(c["text"][:220] + "...", width - 6), "      "))
    print("-" * width)

show(dense_search("What are the two RAG formulations proposed in the paper?"),
     query="What are the two RAG formulations proposed in the paper?")

In [ ]:
# Paraphrase: not one content word here appears in the paper's own phrasing.
show(dense_search("does putting the answer halfway through a long prompt hurt accuracy?", k=3),
     query="does putting the answer halfway through a long prompt hurt accuracy?")

In [ ]:
# Now a query that is mostly one exact string. Watch dense retrieval get vague.
show(dense_search("9%-19% absolute top-20 accuracy", k=3),
     query="9%-19% absolute top-20 accuracy")

### 🔍 How the code works

- `dense_search` is four lines: embed the query with **the same model** used at index time,
  normalise, search, map integer ids back to chunk records. Everything before this step
  existed to make those four lines meaningful.
- `if i != -1` guards the case where FAISS has fewer than `k` vectors and pads with `-1`.
- The score is cosine similarity in `[-1, 1]`. For OpenAI embeddings, **0.3 is a decent match
  and 0.5 is a strong one** — do not expect 0.9. What matters is the *gap* between rank 1 and
  rank 5, not the absolute number.

### ⚙️ Behind the scenes

One embedding call (~15 tokens) plus 180 dot products. The API round trip dominates
completely: ~200 ms of network, ~0.2 ms of maths.

### 🧩 What the three queries showed

| Query | What it tested | Result |
|---|---|---|
| "two RAG formulations" | Vocabulary shared with the text | Works |
| "halfway through a long prompt" | Pure paraphrase, no shared words | Works — this is what embeddings are *for* |
| "9%-19% absolute top-20" | Rare exact strings | Wobbly |

That third row is the entire motivation for Step 5. Embeddings compress meaning, and
compression loses exactly the things that carry no meaning on their own: product codes,
error numbers, surnames, version strings, `9%-19%`. Those are also disproportionately what
users type into a search box.

### ✅ Checkpoint

Ask: **"Retrieval returned five chunks for every one of those queries — including a question
the corpus cannot answer. Why?"**
Answer: nearest-neighbour search always returns `k` neighbours. There is no "no match"; there
is only "the least-bad match." Handling that is the prompt's job, in Step 7.

---

## Step 5 · Hybrid retrieval — BM25 + dense, fused with RRF

### 🎯 What we are going to implement

A BM25 keyword index over the same chunks, then Reciprocal Rank Fusion to merge the two
ranked lists into one.

### 💡 Why we are implementing it

Dense and sparse retrieval fail in **opposite** directions:

| | BM25 (sparse) | Embeddings (dense) |
|---|---|---|
| Matches | Exact terms, weighted by rarity | Meaning, regardless of wording |
| Great at | IDs, codes, names, numbers, jargon | Paraphrase, synonyms, questions |
| Blind to | Synonyms — "car" ≠ "automobile" | The precise token you actually typed |
| Needs training | No | Yes (someone else did it) |

Combining them is not a clever trick; it is the default in Elasticsearch, OpenSearch,
Weaviate, Qdrant and Azure AI Search. On the WANDS benchmark, hybrid scores 0.7497 NDCG
against 0.6983 for BM25 alone and 0.6953 for vectors alone — **better than either, and by
more than the gap between them.**

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(s):
    return re.findall(r"[a-z0-9]+", s.lower())      # crude on purpose: BM25 wants terms

bm25 = BM25Okapi([tokenize(c["text"]) for c in CHUNKS])

def sparse_search(query, k=5):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), CHUNKS[i]) for i in top]

q = "9%-19% absolute top-20 accuracy"
print(">>> BM25"); show(sparse_search(q, k=3), query=q)

In [ ]:
def rrf(*ranked_lists, k=60, top_k=5):
    """Reciprocal Rank Fusion: score(d) = sum over lists of 1 / (k + rank(d)).

    Uses ranks, not scores, so BM25's unbounded numbers and cosine's [-1,1]
    never have to be put on the same scale.
    """
    fused = {}
    for results in ranked_lists:
        for rank, (_score, chunk) in enumerate(results, start=1):
            cid = chunk["id"]
            entry = fused.setdefault(cid, {"score": 0.0, "chunk": chunk})
            entry["score"] += 1.0 / (k + rank)
    best = sorted(fused.values(), key=lambda e: -e["score"])[:top_k]
    return [(e["score"], e["chunk"]) for e in best]

def hybrid_search(query, k=5, pool=20):
    return rrf(dense_search(query, k=pool), sparse_search(query, k=pool), top_k=k)

print(">>> HYBRID"); show(hybrid_search(q, k=3), query=q)

In [ ]:
# Where does each method put the chunk that actually holds the answer?
def positions(query, needle):
    row = {}
    for name, fn in [("dense", dense_search), ("bm25", sparse_search), ("hybrid", hybrid_search)]:
        hits = [i for i, (_, c) in enumerate(fn(query, k=10), start=1)
                if needle.lower() in c["text"].lower()]
        row[name] = hits[0] if hits else None
    return row

probes = [
    ("9%-19% absolute top-20 accuracy",                              "9%-19%"),
    ("does putting the answer in the middle hurt accuracy?",         "U-shaped"),
    ("what are RAG-Sequence and RAG-Token?",                         "RAG-Token"),
    ("which encoder architecture does the retriever use?",           "dual-encoder"),
]
print(f"{'query':52s} {'dense':>6s} {'bm25':>6s} {'hybrid':>7s}   (rank of first relevant, of 10)")
for query, needle in probes:
    r = positions(query, needle)
    fmt = lambda v: "  —  " if v is None else f"{v:^5d}"
    print(f"{query[:50]:52s} {fmt(r['dense']):>6s} {fmt(r['bm25']):>6s} {fmt(r['hybrid']):>7s}")

### 🔍 How the code works

- **`BM25Okapi`** takes tokenized documents and scores a query by term frequency weighted by
  inverse document frequency, normalised by document length. Rare terms in the query that
  appear often in a short chunk score highest — which is precisely why `9%-19%` finds its home.
- **`tokenize` is deliberately crude.** Lowercase, alphanumeric runs. Production systems add
  stemming and stopword removal; the difference is real but small next to having BM25 at all.
- **RRF** is the whole trick and it is one line. Take each list's **rank**, add `1/(60 + rank)`
  across lists. A chunk at rank 1 in both lists scores `1/61 + 1/61`; a chunk at rank 1 in one
  list and absent from the other scores `1/61`. Consensus wins, but a strong single-list result
  still surfaces.
- **Why ranks and not scores?** BM25 returns unbounded positives; cosine returns `[-1, 1]`.
  Any weighted-sum blend needs per-corpus normalisation and a tuned weight. RRF needs neither.
  `k=60` is the value everyone uses, from the 2009 paper; it damps the influence of top ranks
  just enough that one list cannot dominate.
- **`pool=20`** means each retriever proposes 20 candidates and RRF picks the best 5. Fusing
  short lists throws away the agreement signal — always fuse deep, return shallow.

### ⚙️ Behind the scenes

BM25 costs no API calls and negligible time at this scale. In production the inverted index
lives in Elasticsearch or inside your vector DB — but the algorithm you just wrote is the
one running underneath.

### ❓ Questions the client will ask

> *"Should I always use hybrid?"*
> If your users type names, IDs, error codes, or jargon: yes, and it is usually the single
> biggest retrieval win available. If they type natural-language questions about concepts,
> dense alone may be enough. The honest answer, again, is Step 8.

> *"Why 60?"*
> Empirical, from the original RRF paper, and near-universally adopted. Smaller `k` weights
> the top ranks harder. It is worth tuning roughly once and then forgetting.

### ✅ Checkpoint

Ask: **"Which column in that table is best, and what does 'best' mean?"**
Push them: dense wins some rows, BM25 wins others, hybrid is rarely worst. "Best on average
across the queries our users actually ask" is the only definition that survives contact with
production — and it needs a golden set, which we build in Step 8.

---

## Step 6 · Reranking — spend real compute on twenty candidates

### 🎯 What we are going to implement

A cross-encoder reranker: retrieve 20 candidates cheaply, score all 20 properly, keep the
best 4.

### 💡 Why we are implementing it

Everything so far is a **bi-encoder** design: the chunk was embedded months before your query
existed, so the vector cannot possibly know what was asked. It is a lossy summary compared
against another lossy summary.

A **cross-encoder** puts the query and the chunk into one transformer input and reads them
*together*, with full attention between them. It is far more accurate and hopelessly slower —
you cannot run it over 180 chunks per query, let alone a million.

So you use both, and the shape of the pipeline follows directly:

```
180 chunks ──cheap ANN search──► 20 candidates ──expensive cross-encoder──► 4 chunks ──► prompt
             (milliseconds)                        (~50-200 ms)
```

This is **retrieve-and-rerank**, and it is the standard architecture. The first stage
optimises recall — get the answer *somewhere* in the 20. The second optimises precision —
get it to rank 1.

In [ ]:
from sentence_transformers import CrossEncoder

# ~90 MB, downloads once. BAAI/bge-reranker-base is stronger and ~1.1 GB — swap it in
# when you have a GPU and a corpus that deserves it.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)

def rerank(query, candidates, top_k=4):
    """candidates: [(score, chunk), ...] -> the top_k by cross-encoder relevance."""
    pairs = [[query, c["text"]] for _s, c in candidates]
    scores = reranker.predict(pairs)                     # one forward pass per pair
    order = np.argsort(scores)[::-1][:top_k]
    return [(float(scores[i]), candidates[i][1]) for i in order]

In [ ]:
q = "how does the position of the relevant passage affect the model's answer?"

first_stage = hybrid_search(q, k=20, pool=20)
t0 = time.time(); reranked = rerank(q, first_stage, top_k=4); dt = time.time() - t0

print(">>> FIRST STAGE (hybrid, top 4 of 20)"); show(first_stage[:4], query=q)
print(f">>> AFTER RERANKING ({dt*1000:.0f} ms for 20 pairs)"); show(reranked)

In [ ]:
# The useful view: how far did each chunk move?
before = {c["id"]: i for i, (_, c) in enumerate(first_stage, start=1)}
print(f"{'chunk':8s} {'before':>7s} {'after':>6s}  {'source':38s}")
for after_rank, (score, c) in enumerate(reranked, start=1):
    print(f"{c['id']:8s} {before[c['id']]:>7d} {after_rank:>6d}  {c['source']:38s} p{c['page']}")

### 🔍 How the code works

- **`CrossEncoder(...)`** loads a small BERT fine-tuned on MS MARCO to score
  query–passage relevance. Note `max_length=512`: pairs longer than that are **truncated**,
  which is a quiet argument for keeping chunks under ~400 tokens.
- **`reranker.predict(pairs)`** is a batched forward pass — 20 pairs is one batch. The output
  is a raw logit, not a probability. Only the **ordering** is meaningful; do not threshold on
  the absolute value without calibrating first.
- **`top_k=4` from a pool of 20.** The ratio matters more than either number. Too small a pool
  and the reranker has nothing good to promote; too large and you pay for candidates that were
  never plausible. 20–50 in, 3–5 out is the usual band.
- The before/after table is the demo that sells reranking. When a chunk jumps from rank 9 to
  rank 1, that is a chunk the LLM would never have seen.

### ⚙️ Behind the scenes

First run downloads ~90 MB from HuggingFace. After that it is local and free — no API, no
per-query cost, ~50–200 ms on CPU for 20 pairs. This is the cheapest quality improvement in
the pipeline, which is why it is worth a good ten minutes of the session.

### ❓ Questions the client will ask

> *"Why not skip the vector search and cross-encode everything?"*
> Do the arithmetic with them: 180 chunks × ~10 ms ≈ 2 seconds per query, and that is a toy
> corpus. A million chunks is three hours. Bi-encoders are precomputable and cross-encoders are
> not — that single asymmetry is why the two-stage pipeline exists at all.

> *"Open model or an API like Cohere Rerank?"*
> Cohere Rerank 4 is the lowest-friction managed option; BGE-reranker-v2-m3 is Apache-2.0,
> multilingual, ~50–100 ms on GPU, and free to self-host. The one we loaded is the small
> English workhorse — right for a laptop, right for a lesson.

> *"Does reranking let me use worse chunks or a cheaper embedding model?"*
> Partly, and that is a real deployment lever. It cannot recover an answer the first stage
> never retrieved. **Recall lost in stage one is lost forever.**

### ✅ Checkpoint

Ask: **"Chunk c0091 moved from rank 9 to rank 1. What did the cross-encoder know that the
embedding did not?"**
Answer: it saw the question. The embedding was computed before the question existed and had to
summarise the chunk for *all possible* questions at once.

---

## Step 7 · Prompt construction — turning chunks into a grounded answer

### 🎯 What we are going to implement

A context block with labelled sources, a system prompt that forbids ungrounded answers, a
token budget, and the generation call.

### 💡 Why we are implementing it

Retrieval finds text. **The prompt decides what counts as an answer.** Four things are
load-bearing here, and each one prevents a specific, common failure:

| Design choice | Failure it prevents |
|---|---|
| "Use only the context" | The model answering from its pretraining, confidently and unverifiably |
| An explicit escape hatch | The model inventing something rather than admitting the context is silent |
| Source labels **inside** the text | Citations that cannot exist, because the model never saw the metadata |
| A token budget | A prompt that grows until it is slow, expensive, and — per the third paper — *less* accurate |

That last row is worth pausing on. "Lost in the Middle" is in our corpus for a reason: model
accuracy follows a **U-shaped curve** against the position of the relevant passage. Evidence
at the start or the end is used well; evidence buried in the middle of a long context is
often ignored. **More context is not more accuracy.** It is the empirical justification for
everything we built in Steps 5 and 6 — the whole point of reranking is to put the best chunk
at position 1.

In [ ]:
SYSTEM = """You answer questions using only the provided context.

Rules:
- Use only information in the CONTEXT block. Do not use prior knowledge.
- Cite the sources you used with their bracket labels, e.g. [S1] or [S2][S3].
- If the context does not contain the answer, reply exactly:
  "The provided documents do not contain the answer to that question."
- Be concise. Two or three sentences unless the question demands more."""

def build_context(results, budget_tokens=1600):
    """Format ranked chunks into a labelled context block, respecting a token budget."""
    blocks, used, spent = [], [], 0
    for i, (_score, c) in enumerate(results, start=1):
        block = f"[S{i}] ({c['source']}, page {c['page']})\n{c['text']}"
        cost = ntok(block)
        if spent + cost > budget_tokens:
            break                       # stop cleanly rather than truncating mid-chunk
        blocks.append(block); used.append(c); spent += cost
    return "\n\n".join(blocks), used, spent

In [ ]:
def generate(question, results, budget_tokens=1600):
    context, used, spent = build_context(results, budget_tokens)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
    ]
    resp = client.chat.completions.create(
        model=CHAT_MODEL, messages=messages, temperature=0,
    )
    return resp.choices[0].message.content, used, spent

question = "What are the two RAG formulations, and how do they differ?"
answer, used, spent = generate(question, rerank(question, hybrid_search(question, k=20, pool=20)))

print("Q:", question)
print("\nA:", textwrap.fill(answer, 92))
print("\ncontext tokens:", spent)
for i, c in enumerate(used, start=1):
    print(f"  [S{i}] {c['source']} p{c['page']}  ({c['id']})")

In [ ]:
# The two questions that matter more than the one that works.
for q in ["What is the U-shaped curve reported for long-context models?",
          "What learning rate did the authors use to fine-tune YOLOv8?"]:
    ans, used, spent = generate(q, rerank(q, hybrid_search(q, k=20, pool=20)))
    print("Q:", q)
    print("A:", textwrap.fill(ans, 92))
    print("   retrieved:", [f"{c['source'].split(' (')[0]} p{c['page']}" for c in used])
    print("-" * 96)

### 🔍 How the code works

- **`[S1]`, `[S2]` labels are inside the string the model reads.** This is the mechanism that
  makes citation possible at all — the model cannot cite a Python dictionary it never saw.
  The label maps back to `used[i-1]`, so your UI can turn `[S1]` into a real link to page 6.
- **`build_context` stops at the budget instead of truncating.** A half-chunk is a chunk whose
  last sentence lies. Dropping it whole is always safer.
- **The refusal string is exact**, so you can detect it in code — `if answer.startswith("The
  provided documents")` is how you route to a fallback, a human, or a web search.
- **`temperature=0`** for grounded Q&A. You want the same answer twice from the same context.
- Ordering is not accidental: `build_context` emits results **in rank order**, so the
  reranker's best chunk lands at the top of the prompt, where the model reads it best.

### ⚙️ Behind the scenes

Watch the second question. Nothing in three retrieval papers mentions YOLOv8 — but retrieval
still returned four chunks, because nearest-neighbour search always does. The refusal comes
**entirely from the system prompt**. Say this plainly to the client:

> **Retrieval finds. The prompt decides.**

Cost per question: one embedding call (~15 tokens) plus one chat call (~1,700 in, ~80 out).
On `gpt-5.6-luna` (\$1 in / \$6 out per 1M tokens) that is about **0.2 cents** — and roughly
**1/30th** of what stuffing
all three papers into the prompt would cost, per question, forever.

### ❓ Questions the client will ask

> *"Can I trust the citations?"*
> Not blindly — a model can attach `[S2]` to a claim that came from `[S1]`, or occasionally to
> a claim that came from nowhere. Two mitigations: keep chunks small enough that a human can
> check quickly, and measure citation faithfulness explicitly (Session 37, RAGAS).

> *"Why not just pass 20 chunks and let the model sort it out?"*
> Slower, ~5× the cost, and — per the paper in our own corpus — likely *less* accurate.
> This is the single most counter-intuitive fact in RAG and it is worth landing hard.

### ✅ Checkpoint

Ask: **"Which single line in `SYSTEM` produced the refusal on the YOLOv8 question?"**
Then follow up: **"What should a production system do when it sees that string?"** — log it,
show the user a fallback, and add the question to the evaluation set. Refusals are the most
valuable telemetry a RAG system produces.

---

## Step 8 · Evaluation — did any of that actually help?

### 🎯 What we are going to implement

A small golden set of questions with known answers, two retrieval metrics (**hit rate** and
**MRR**), and an ablation table comparing dense vs BM25 vs hybrid vs hybrid+rerank.

### 💡 Why we are implementing it

Every decision today was a judgement call: chunk size 320, overlap 48, pool 20, top-k 4,
RRF over weighted sum, cross-encoder or not. Without measurement they are **opinions**, and
the next change you make is as likely to hurt as to help.

Two metrics carry most of the weight:

- **Hit rate@k** — in what fraction of queries does at least one relevant chunk appear in the
  top k? *"Did we find it at all?"*
- **MRR@k** — mean of `1/rank` of the first relevant chunk. *"How near the top did we find it?"*
  This is the metric that moves when you add a reranker, because reranking changes order, not
  membership.

Rough industry targets, for calibration: recall@5 ≥ 0.80 on ordinary corpora, MRR ≥ 0.6 feels
"snappy and correct" to a user, NDCG@5 ≥ 0.8 on head queries once a reranker is in place.

**Retrieval evaluation first, generation evaluation second.** If the right chunk never
arrives, no prompt can rescue the answer — so measure the stage you can actually fix.

In [ ]:
# A golden set: question -> strings that MUST appear in a chunk for it to count as relevant.
# Ten questions is not a benchmark. It is enough to stop you shipping a regression.
GOLDEN = [
    ("What are the two RAG formulations proposed in the paper?",        ["RAG-Sequence", "RAG-Token"]),
    ("What is the non-parametric memory in the RAG model?",             ["dense vector index of Wikipedia"]),
    ("How much does DPR beat BM25 on top-20 retrieval accuracy?",       ["9%-19%"]),
    ("What encoder architecture does the dense retriever use?",         ["dual-encoder", "dual- encoder"]),
    ("What shape is the curve when the relevant passage moves?",        ["U-shaped"]),
    ("What are primacy and recency bias in long contexts?",             ["primacy", "recency"]),
    ("Which similarity function does DPR use to compare vectors?",      ["dot product", "inner product"]),
    ("Is retrieval-augmented generation trained end-to-end?",           ["end-to-end", "end to end"]),
    ("Which datasets were used to evaluate open-domain QA?",            ["Natural Questions", "TriviaQA"]),
    ("How does model performance change as the context gets longer?",   ["degrades", "decrease"]),
]

def is_relevant(chunk, needles):
    return any(n.lower() in chunk["text"].lower() for n in needles)

# Sanity check FIRST: a question whose answer is in no chunk is a broken label, not a hard query.
for q, needles in GOLDEN:
    n = sum(is_relevant(c, needles) for c in CHUNKS)
    flag = "  <-- LABEL PROBLEM" if n == 0 else ""
    print(f"{n:3d} chunks contain the answer to: {q[:58]}{flag}")

In [ ]:
def evaluate(retrieve_fn, k=5, name=""):
    """Hit rate@k and MRR@k over the golden set."""
    hits, rrs = [], []
    for q, needles in GOLDEN:
        ranked = [c for _s, c in retrieve_fn(q, k)]
        flags = [is_relevant(c, needles) for c in ranked[:k]]
        hits.append(any(flags))
        rrs.append(1.0 / (flags.index(True) + 1) if any(flags) else 0.0)
    return {"name": name, "hit_rate": float(np.mean(hits)), "mrr": float(np.mean(rrs))}

def rerank_pipeline(q, k=5):
    return rerank(q, hybrid_search(q, k=25, pool=25), top_k=k)

CONFIGS = [
    ("dense only",        dense_search),
    ("BM25 only",         sparse_search),
    ("hybrid (RRF)",      hybrid_search),
    ("hybrid + reranker", rerank_pipeline),
]

rows = [evaluate(fn, k=5, name=name) for name, fn in CONFIGS]

print(f"{'configuration':22s} {'hit rate@5':>11s} {'MRR@5':>8s}")
print("-" * 43)
for r in rows:
    print(f"{r['name']:22s} {r['hit_rate']:>11.2f} {r['mrr']:>8.3f}")

### 🔍 Read the table with the client — do not just show it

Ask them what they expect **before** you run the cell. Then interpret what actually happened:

- **BM25 vs dense** — they usually trade places question by question. Neither is "the good one."
- **Hybrid** — should match or beat both on hit rate. It is a recall play: two chances to find
  the chunk.
- **+ reranker** — expect the largest jump in **MRR** with a smaller change in hit rate. That is
  exactly what a reranker is: it does not find new chunks, it reorders the ones you have. If
  hit rate is already 1.0, MRR is the only number left that can move.
- **If reranking does not help here**, say so out loud. On ten easy questions over ~180 chunks,
  the first stage is often good enough. Rerankers earn their keep on large, noisy corpora with
  many near-duplicates — and pretending otherwise teaches the client to cargo-cult.

### 🔍 How the code works

- **Substring labels** are the pragmatic trick. Labelling chunk *ids* would be more precise but
  breaks the moment you change the chunk size — which is exactly the experiment you most want
  to run. Substrings survive re-chunking.
- **The sanity check runs first, on purpose.** A question with zero matching chunks scores 0 for
  every configuration and quietly drags all your averages down. Validate labels before trusting
  metrics.
- **`flags.index(True) + 1`** is the 1-based rank of the first relevant chunk; its reciprocal is
  the RR for that query. Average over queries and you have MRR.
- **Same `k`, same golden set, one variable at a time.** That is what makes it an ablation
  rather than four unrelated numbers.

### ⚙️ Behind the scenes

One embedding call per query per configuration — about 40 calls, well under a cent. Cache the
query embeddings and it is 10. At production scale, evaluation cost is a real budget line, which
is why golden sets stay in the low hundreds and get curated rather than grown.

### ❓ Questions the client will ask

> *"Ten questions isn't a benchmark."*
> Correct, and say so first. Ten questions catches *regressions*, which is 90% of the value.
> Aim for 50–200 real user questions, and the best source is your own logs — especially the
> refusals from Step 7.

> *"How do I evaluate the generated answer, not just retrieval?"*
> Faithfulness (is every claim supported by the context?), answer relevance, and context
> precision — all typically scored by an LLM judge. That is **RAGAS**, and it is Session 37.
> Retrieval metrics come first because retrieval is what you can actually fix.

### ✅ Checkpoint

Ask: **"Change chunk size from 320 to 128 and this table changes. Which number would you look
at, and what would make you keep the change?"**
That question is the whole discipline: a knob, a metric, a decision.

---

## Step 9 · The whole pipeline in one function — and three failure drills

### 🎯 What we are going to implement

Collapse the seven stages into a single `rag(question)` function, then deliberately break it
three ways.

### 💡 Why we are implementing it

Two reasons. First, the client should see that the entire session compresses to about fifteen
readable lines — the complexity was in the *choices*, not the code. Second, a pipeline you
have only seen succeed is a pipeline you cannot debug. Breaking it on purpose, with a warning
first, is the fastest way to teach the failure signatures.

In [ ]:
def rag(question, pool=25, top_k=4, budget=1600, verbose=True):
    """The full pipeline: retrieve -> fuse -> rerank -> ground -> answer."""
    candidates = hybrid_search(question, k=pool, pool=pool)   # dense + BM25, fused by RRF
    top        = rerank(question, candidates, top_k=top_k)    # cross-encoder
    answer, used, spent = generate(question, top, budget)     # grounded generation
    if verbose:
        print("Q:", question)
        print("\nA:", textwrap.fill(answer, 92))
        print("\nsources:")
        for i, c in enumerate(used, start=1):
            print(f"  [S{i}] {c['source']}, page {c['page']}  ({c['id']}, {c['tokens']} tok)")
        print(f"\ncontext: {spent} tokens from {len(candidates)} candidates")
    return answer, used

_ = rag("Why does retrieval-augmented generation reduce hallucination compared to a "
        "parametric-only model?")

In [ ]:
# ── Drill 1: the context is right, the question is unanswerable from it.
_ = rag("What is the capital of Australia?")

In [ ]:
# ── Drill 2: the prompt loses its grounding rule. Run this once, then never again.
LOOSE = "You are a helpful assistant. Use the context if it helps."

def rag_ungrounded(question):
    top = rerank(question, hybrid_search(question, k=25, pool=25), top_k=4)
    context, _used, _spent = build_context(top)
    resp = client.chat.completions.create(
        model=CHAT_MODEL, temperature=0,
        messages=[{"role": "system", "content": LOOSE},
                  {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}],
    )
    return resp.choices[0].message.content

q = "What is the capital of Australia?"
print("GROUNDED  :", textwrap.fill(rag(q, verbose=False)[0], 88))
print()
print("UNGROUNDED:", textwrap.fill(rag_ungrounded(q), 88))

In [ ]:
# ── Drill 3: the alignment bug. Shift the chunk mapping by one and watch nothing crash.
saved = CHUNKS[:]
CHUNKS[:] = CHUNKS[1:] + CHUNKS[:1]          # off-by-one, the classic

ans, used = rag("What are the two RAG formulations proposed in the paper?", verbose=False)
print("ANSWER  :", textwrap.fill(ans, 88))
print("CITED   :", [f"{c['source'].split(' (')[0]} p{c['page']}" for c in used])
print("\nNo exception. No warning. The citations are simply wrong.")

CHUNKS[:] = saved                            # put it back before anything else runs
print("restored:", len(CHUNKS), "chunks")

### 🔍 What the three drills teach

| Drill | Symptom | Where the fix lives |
|---|---|---|
| Unanswerable question | Confident refusal, correct behaviour | The system prompt (Step 7) |
| Ungrounded prompt | A fluent, plausible, unsourced answer | The system prompt (Step 7) |
| Off-by-one alignment | Right answer, **wrong citations**, no error | The `assert` in Step 3 |

Drill 2 is the one clients remember. The ungrounded model answers "Canberra" perfectly well —
and that is precisely the problem, because you now have no idea which of its answers came from
your documents and which came from its pretraining. In a medical or legal deployment that
distinction is the entire product.

Drill 3 is the one engineers remember. Every stage ran, no exception was raised, the answer was
even correct — and the citations pointed at the wrong pages. **The most dangerous RAG bugs do
not raise exceptions.** They are found by asserts you wrote in advance and by evaluation sets
you maintain, not by error logs.

### ✅ Checkpoint

Ask: **"Which of these three would take longest to notice in production, and what would you add
today to catch it?"**

---

## Step 10 · Recap, the knobs table, and exercises

### What we built

```
 3 PDFs
   │
   ├─ Step 1  INGEST     pypdf -> clean() -> {source, page, text}         50 pages
   ├─ Step 2  CHUNK      recursive, 320 tokens, 48 overlap                ~180 chunks
   ├─ Step 3  INDEX      embed (batched) -> FAISS IndexFlatIP + JSON      ~1.1 MB
   │
   ▼
question
   │
   ├─ Step 4  DENSE      embed query -> top 25                 ─┐
   ├─ Step 5  SPARSE     BM25 -> top 25                         ├─ RRF fuse
   ├─ Step 6  RERANK     cross-encoder over 25 -> top 4        ─┘
   ├─ Step 7  PROMPT     labelled context + grounding rules -> gpt-5.6-luna
   │
   ▼
answer + citations
   │
   └─ Step 8  EVALUATE   hit rate@5, MRR@5 over a golden set
```

### Every knob you touched today

| Knob | We used | Turn it up when | Turn it down when |
|---|---|---|---|
| Chunk size | 320 tokens | Answers need surrounding context | Questions are precise and factoid |
| Overlap | 48 tokens (15%) | Answers straddle boundaries | Index cost matters and eval says it's free |
| First-stage pool | 25 | Reranking is on and recall is short | Latency matters |
| Final top-k | 4 | Answers need to synthesise several sources | Precision matters more than coverage |
| Context budget | 1,600 tokens | Never, casually — remember the U-curve | Latency or cost is the complaint |
| RRF `k` | 60 | You want deep ranks to matter more | You want the top ranks to dominate |
| Reranker | MiniLM-L6 | Corpus is large, noisy, near-duplicate-heavy | Latency budget is tight |

**None of these has a correct value. Each has a measurable one.**

### Seven things worth remembering

1. **The chunk is the unit of retrieval.** It is the highest-leverage decision in the pipeline
   and it is made before any clever component runs.
2. **Ingestion damage is silent.** Nothing throws; the quality just quietly caps out.
3. **Dense and sparse retrieval fail in opposite directions.** Hybrid + RRF is cheap and is the
   production default for a reason.
4. **Bi-encoders find, cross-encoders rank.** Recall lost in stage one is lost forever.
5. **More context is not more accuracy** — the U-shaped curve is in a paper in our own corpus.
6. **Retrieval finds; the prompt decides.** Grounding rules and the refusal path are code, not
   politeness.
7. **Every knob needs a number attached.** A golden set of even ten questions beats an opinion.

### Exercises

1. **Chunk-size sweep.** Re-run Steps 2–3 with `size` of 128, 320 and 640, and rebuild the
   Step 8 table for each. Which size wins on MRR, and does the winner change if you turn the
   reranker off?
2. **Kill the overlap.** Set `overlap=0` and re-evaluate. Did the 15% extra storage buy anything
   on *this* corpus? (Recall the 2026 finding that it often does not.)
3. **Add a fourth paper** of your own choosing and check that no golden answer changes. If one
   does, you have discovered corpus interference — worth ten minutes of discussion.
4. **Metadata filtering.** Add a `paper=` argument to `hybrid_search` that restricts retrieval to
   one source. Compare "what did DPR report?" with and without the filter.
5. **Query rewriting.** Before retrieving, ask `gpt-5.6-luna` to expand the question into three
   paraphrases, retrieve for each, and RRF all three lists together. Does the Step 8 table move?
6. **Grow the golden set to 25** using questions the client invents. Watch how much less
   flattering the numbers become — that is the point.

### Where this goes next

| Session | Topic | What it adds to today |
|---|---|---|
| 34 | RAG Pipeline Lab II | Metadata filters, query rewriting, multi-query, streaming |
| 35 | Advanced RAG | HyDE, parent-document retrieval, contextual compression |
| 36 | Multimodal RAG | Retrieving over figures and tables, not just prose |
| 37 | RAG Evaluation (RAGAS) | Automated faithfulness and answer-relevance scoring |
| 38 | Milestone | The full system, on the client's own documents |

---

*Session 33 · RAG Pipeline Lab I · Phase 9 — RAG Systems*